In [1]:
import pandas as pd
from collections import defaultdict
 
 
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/titanic.csv"
df = pd.read_csv(url)

In [2]:
#A case
val = df.Survived.value_counts(normalize=True)
print(val)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64


Pregunta A (Sumarización Categórica): Usen la función .value_counts(normalize=True) en la columna Survived (0 = Murió, 1 = Sobrevivió). ¿Cuál es la tasa de supervivencia global del barco expresada en porcentaje?

De acuerdo a la funcion, el 38% de las personas del barco sobrevivieron al accidente.

In [3]:
#B Case
bVal = df.groupby('Sex')['Survived'].mean()
print(bVal)

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64


Pregunta B : El famoso código marítimo era "mujeres y niños primero". Comprobemos esto matemáticamente. Ejecuten una agrupación por la columna de género y calculen el promedio de la columna Survived. ¿Qué porcentaje exacto de mujeres sobrevivió en contraste con los hombres?

El 74% de mujeres sobrevivieron al accidente en contraste al 18% de hombres

In [4]:
#C Case
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)

iqr = Q3 - Q1

max = Q3 + (1.5 * iqr)

print(max)
countMax = 0
map_val = defaultdict(int)

for val in df.itertuples():
    if val.Fare > max:
        countMax += 1
        map_val[val.Pclass] += 1

print("Count High : " , countMax)
print("Map : ", map_val)

65.6344
Count High :  116
Map :  defaultdict(<class 'int'>, {1: 104, 2: 5, 3: 7})


Pregunta C: El director financiero nota que la tarifa máxima (Fare) cobrada fue de más de 500 libras, mientras que el promedio ronda las 32. Sospecha de outliers. Utilicen las herramientas de dispersión en Pandas para confirmarlo:
Calculen el Cuartil 1 (25%) y el Cuartil 3 (75%) de la columna Fare usando el método .quantile([0.25, 0.75]).
Calculen matemáticamente el Rango Intercuartílico (IQR = Q3 - Q1).
Calculen el límite superior aceptable (Q3 + 1.5 * IQR). Escriban una línea de código para filtrar el DataFrame y descubrir exactamente cuántos pasajeros pagaron una tarifa por encima de ese límite matemático. ¿A qué clase (Pclass) pertenecían la mayoría de ellos?

Fueron 116 pasajeros que pagaron una tarifa mayor y 104 de ellos eran de la clase 1

In [5]:
#D Case
avg = df.Fare.mean()
median = df.Fare.median()

print("Average : ", avg)
print("Median : ", median)


Average :  32.204207968574636
Median :  14.4542


Pregunta D: Calculen la media y la mediana de la columna Fare. Notarán una diferencia enorme entre ambos valores. Matemáticamente, ¿qué significa que la media sea tan superior a la mediana? Si en el futuro utilizamos un algoritmo basado en distancias euclidianas (como K-Nearest Neighbors) sin escalar previamente esta variable, ¿cómo afectará esta asimetría al aprendizaje del modelo?

Significa que hay un desbalance de los valores hacia el lado mayor o derecha creando una distribucion sesgada, esto en un futuro podrias crear una asimetria en donde se tomarian valoires muy diferenciales que no relefejan como interactuan los datos entre si, teniendo un peso mayor al que realmente significa.


In [6]:
## E Case
proportion = df["Survived"].value_counts(normalize=True)
print(proportion)

n = 150
not_survived = round(proportion[0] * n)
survived = round(proportion[1] * n)

print(not_survived)
print(survived)

not_survived_people = df[df["Survived"] == 0].sample(
    n = not_survived,
    random_state =42
)

survived_people = df[df["Survived"] == 1].sample(
    n = survived,
    random_state = 42
)


muestra = pd.concat([not_survived_people, survived_people])
print(muestra)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64
92
58
     PassengerId  Survived  Pclass  \
312          313         0       2   
124          125         0       1   
783          784         0       3   
167          168         0       3   
772          773         0       2   
..           ...       ...     ...   
237          238         1       2   
190          191         1       2   
513          514         1       1   
166          167         1       1   
869          870         1       3   

                                                Name     Sex   Age  SibSp  \
312            Lahtinen, Mrs. William (Anna Sylfven)  female  26.0      1   
124                      White, Mr. Percival Wayland    male  54.0      0   
783                           Johnston, Mr. Andrew G    male   NaN      1   
167  Skoog, Mrs. William (Anna Bernhardina Karlsson)  female  45.0      1   
772                                Mack, Mrs. (Mary)  female  57.0      0   
..   

Pregunta E: En la semana 2 vimos que el muestreo aleatorio simple es peligroso. El objetivo es entrenar un modelo que prediga la supervivencia (Survived), pero las clases están desbalanceadas. Escriban el código en Pandas para extraer una muestra de exactamente 150 pasajeros garantizando que la proporción de sobrevivientes y no sobrevivientes en la muestra sea idéntica a la de la base de datos completa. ¿Qué sesgo evitan al hacer esto?

Evitamos tener todos los los valores en no survided y tener un sesgo que nos diga que nadie sobrevivio cuando la realidad, es que un 30% sobrevivieron al accidente, por lo que la tener este tipo de muestreo aseguramos estabilidad y coherencia con la base de datos original, evitando los sesgos anteriormente mencionados.

In [ ]:
### Case F
print(df.groupby('Survived')['Age'].mean())

Survived
0    30.626179
1    28.343690
Name: Age, dtype: float64


Pregunta F: Si ejecutan df.groupby('Survived')['Age'].mean(), Pandas calculará el promedio de edad de los que vivieron y los que murieron. Sin embargo, por defecto, Pandas ignora los valores NaN al calcular la media. Si resulta que la gran mayoría de las edades faltantes (NaN) pertenecían a pasajeros de 3ra clase que murieron, ¿qué sesgo estadístico estamos introduciendo involuntariamente en el resultado de esa función y cómo afectaría la inferencia de nuestro modelo?. 

Lo que pasaria es que al momento de hacer las predicciones, el modelo podria llegar a inferir que no existian pasajeros en la tercera clase ya que pandas los ignora, haciendo que el modelo no pudiera interpretar la clase debido a que no habria la suficiente informacion para clasificar algun posible usuario dentro de esa clase, por lo que el modelo terminaria sesgado y no podria obtener una respuesta clara, una solucion seria utilizar la media o mediana para cubrir los NaN en las edades de los pasajeros.

In [ ]:
### Case G
print("Max quartil : ", max)

Max quartil :  65.6344


Pregunta G: Con el cálculo del IQR en la Pregunta C, determinaron que los boletos de 512 libras son outliers matemáticos. En Machine Learning, los valores atípicos pueden representar errores de captura (ruido que aumenta el error irreducible) o casos especiales válidos (señal). Investigando la naturaleza de un barco de lujo, ¿deberíamos eliminar estas filas con .drop() antes de entrenar nuestro modelo? Justifiquen su respuesta arquitectónica.

Considero que no, ya que no son errores de captura, si nos enfocamos en el contexto de un barco de lujo de aquella epoca, es totalmente valido que personas que pertenecieran a la primera clase pagaron un precio sumamente elevado, por lo que este dato nos podria ayudar a saber la clasificacion de un pasajero, cone sto si mas pagaron por el podriamos determinar su prioridad de evacuación, etc.


In [9]:
print(df['Name'])

0                                Braund, Mr. Owen Harris
1      Cumings, Mrs. John Bradley (Florence Briggs Th...
2                                 Heikkinen, Miss. Laina
3           Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                               Allen, Mr. William Henry
                             ...                        
886                                Montvila, Rev. Juozas
887                         Graham, Miss. Margaret Edith
888             Johnston, Miss. Catherine Helen "Carrie"
889                                Behr, Mr. Karl Howell
890                                  Dooley, Mr. Patrick
Name: Name, Length: 891, dtype: object


Pregunta H. Observen la columna Name. Es texto libre (dato no estructurado), pero contiene títulos ocultos como "Mr.", "Mrs.", "Miss." o "Master.". Si lograran extraer ese título usando expresiones regulares en Pandas, podrían hacer un .groupby('Titulo')['Age'].median(). ¿Por qué imputar las edades faltantes basándose en la mediana del "Título" (ej. "Master" = niño, "Mr" = adulto) sería estadísticamente superior y reduciría el error de nuestro futuro modelo, en comparación con usar la mediana global?

Esto se debe a que conocemos por certeza a que el pasajero puede ser parte de una clase que se divida entre rangos (niño, adultos), y como se vio anteriormente que la mayoria de los pasarejos osilaban entre los 20 y 35 años, por lo que si usamos la media o mediana podriamos tener un sesgo si hay muchos valores vacios en el datataset, teniendo como consecuencia no poder distinguir entre niños,ya que pudieran tener una edad mayor, por lo que con el TItulo podriamos tener mas certeza a que edad podria tener el pasajero. haciendo nuestro modelo mas acertado.

In [10]:
survived_variance = df['Survived'].var()
print("Varianza de Supervivencia", survived_variance)

Varianza de Supervivencia 0.23677221654749742


Pregunta I: Calculen la varianza de la columna Survived. Dado que es una variable categórica codificada como 0 y 1, el resultado numérico estará cerca de $0.23$. Matemáticamente, ¿qué significaría si la varianza de esta variable fuera exactamente 0.0? ¿Qué pasaría si intentan entrenar un algoritmo de clasificación con un dataset donde la variable de respuesta tiene varianza 0.0?

Significaria que no existiria otra clase, haciendo que todos los valores de la columna pertenezcan a un solo valor, por lo que si intentamos entrenar un modelo con esta varianza, habria un sesgo en la desicion, haciendo que no pudiera diferenciar entre dos tipos de pasajeros, ademas de que no existiria la opcion de hacer split o dividir por diferentes clases si es necesario gastanto recursos y tiempo en la ejecucion.


In [11]:
print(df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count())

Pclass  Sex     Embarked
1       female  C            43
                Q             1
                S            48
        male    C            42
                Q             1
                S            79
2       female  C             7
                Q             2
                S            67
        male    C            10
                Q             1
                S            97
3       female  C            23
                Q            33
                S            88
        male    C            43
                Q            39
                S           265
Name: PassengerId, dtype: int64


Pregunta J: Ejecuten una agrupación por tres niveles al mismo tiempo y cuenten cuántos pasajeros hay en cada subgrupo: df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count(). Notarán que algunos subgrupos tienen 1 o 2 pasajeros. Si un algoritmo intenta extraer reglas de probabilidad de grupos tan pequeños, se enfrentará a la "Maldición de la Dimensionalidad" (Curse of Dimensionality). ¿Qué fenómeno perjudicial (sobreajuste o subajuste) ocurrirá inevitablemente si dejamos que el modelo aprenda reglas basadas en esos grupos de 1 solo pasajero?

Lo que pasara sera un sobreajuste, ya que si notamos hay varios subgrupos que no estan dando informacion relevante, pero si los contamos por cada pasajero y tienen valores de 1 o 2 careciendo de significancia, se creara bastante ruido dejando los datos muy dispersos y conflictuando el split o decisiones que toman los algoritmos para pdoier aprender un patron para la prediccion.

